# Querying Dataset Metadata with the GDEX API

---

## Overview

GDEX exposes both file-level and dataset-level metadata for each dataset through dedicated endpoints. This notebook demonstrates how to retrieve and use these metadata fields.

1. Retrieve the dataset abstract and acknowledgement
2. Inspect spatial and temporal coverage
3. List variables and data formats
4. Check dataset volume and related datasets

## Prerequisites

| Concepts | Importance | Notes |
| --- | --- | --- |
| api_intro | Necessary | GDEX API base URL and response envelope |
| [requests library](https://requests.readthedocs.io) | Necessary | |
| pandas | Helpful | Tabular display of results |

- **Time to learn**: 20 minutes

## Documentation

Full endpoint documentation can be found **[here](https://gdex.ucar.edu/api/documentation//#/Dataset-level%20Metadata)**

---

## Imports

In [1]:
import requests
import json
import pandas as pd

## Dataset Abstract

`GET /datasets/{dsid}/abstract/` returns cleaned text, extracted URLs, and any advisory notes.

In [26]:
BASE_URL = "https://gdex.ucar.edu/api"
DSID = "d084001"  # GFS 0.25 degree forecasts

abstract = requests.get(f"{BASE_URL}/datasets/{DSID}/abstract/").json()
print(abstract["data"]["abstract"][:500])


The NCEP operational Global Forecast System analysis and forecast grids are on a 0.25 by 0.25 global latitude longitude grid. Grids include analysis and forecast time steps at a 3 hourly interval from 0 to 240, and a 12 hourly interval from 240 to 384. Model forecast runs occur at 00, 06, 12, and 18 UTC daily. For real-time data access please use the NCEP data server. s3.amazonaws.com/index.html">https://noaa-gfs-bdp-pds.s3.amazonaws.com/index.html). Therefore, the RDA will stop updating this da


:::{tip}
All GDEX API responses share a common envelope: `status`, `http_response`, `error_messages`, `contact`, and `data`. Check `response["error_messages"]` if a call returns unexpected results.
:::


## Spatial and Temporal Coverage

`GET /datasets/{dsid}/spatial_coverage/` returns bounding box, resolution, and grid dimensions. `GET /datasets/{dsid}/temporal/` returns start/end dates and sub-group time ranges.


In [18]:
spatial  = requests.get(f"{BASE_URL}/datasets/{DSID}/spatial_coverage/").json()
temporal = requests.get(f"{BASE_URL}/datasets/{DSID}/temporal/").json()
bounds   = spatial["data"]["spatial_coverage"]["bounds"]
print(bounds)
print(f"Start data: {temporal['data']['temporal']['start_date']}")
print(f"End data:   {temporal['data']['temporal']['end_date']}")


{'north': 90.0, 'south': -90.0, 'east': 180.0, 'west': -180.0, 'lat_units': 'degrees north', 'lon_units': 'degrees east'}
Start data: 2015-01-15 00:00 +0000
End data:   2026-07-03 12:00 +0000


## Variables and detailed metadata

`GET /paramsummary/{dsid}` returns all variables. `GET /metadata/{dsid}/` gives detailed inventory that you can filter.


In [32]:
print(f"{BASE_URL}/paramsummary/{DSID}/")
print(f"{BASE_URL}/metadata/{DSID}/")
variables = requests.get(f"{BASE_URL}/paramsummary/{DSID}/").json()
metadata   = requests.get(f"{BASE_URL}/metadata/{DSID}/").json()
all_metadata = metadata["data"]["data"]
df = pd.DataFrame(all_metadata)
df


https://gdex.ucar.edu/api/paramsummary/d084001/
https://gdex.ucar.edu/api/metadata/d084001/


,param,param_description,start_date,end_date,native_format,gridproj,griddef,level,units,standard_name,GCMD_uuid,product,levels
0,T MAX,Maximum temperature,201501160000,202606181800,WMO_GRIB2,latLon,1440:721:90N:0E:90S:359.75E:0.25:0.25,None,K,maximum_air_temperature,5164162a-60eb-4c94-a0f0-2caaa3bb1754,6-hour Maximum (initial+24 to initial+30),"[{'units': 'm', 'level_value': '2', 'level_des..."
1,CICEP,Categorical ice pellets (yes=1; no=0),201906171800,202606222100,WMO_GRIB2,latLon,1440:721:90N:0E:90S:359.75E:0.25:0.25,None,NaN,NaN,NaN,129-hour Forecast,"[{'level_value': '0', 'level_description': 'Gr..."
2,CNWAT,Plant canopy surface water,202104031200,202606291200,WMO_GRIB2,latLon,1440:721:90N:0E:90S:359.75E:0.25:0.25,None,kg m^-2,NaN,NaN,288-hour Forecast,"[{'level_value': '0', 'level_description': 'Gr..."
3,V-GWD,Meridional flux of gravity wave stress,201906240600,202606291200,WMO_GRIB2,latLon,1440:721:90N:0E:90S:359.75E:0.25:0.25,None,N m^-2,NaN,7e6f7c15-32e7-4b6e-bd35-7bff4bc03caf,6-hour Average (initial+282 to initial+288),"[{'level_value': '0', 'level_description': 'Gr..."
4,FRICV,Frictional velocity,202204161200,202606290600,WMO_GRIB2,latLon,1440:721:90N:0E:90S:359.75E:0.25:0.25,None,m s^-1,NaN,NaN,282-hour Forecast,"[{'level_value': '0', 'level_description': 'Gr..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
11415,U FLX,"Momentum flux, u-component",201501160000,202606181800,WMO_GRIB2,latLon,1440:721:90N:0E:90S:359.75E:0.25:0.25,None,N m^-2,downward_eastward_momentum_flux_in_air,NaN,6-hour Average (initial+24 to initial+30),"[{'level_value': '0', 'level_description': 'Gr..."
11416,V FLX,"Momentum flux, v-component",201501160000,202606181800,WMO_GRIB2,latLon,1440:721:90N:0E:90S:359.75E:0.25:0.25,None,N m^-2,downward_northward_momentum_flux_in_air,NaN,6-hour Average (initial+24 to initial+30),"[{'level_value': '0', 'level_description': 'Gr..."
11417,VW SH,Vertical speed shear,201501160600,202606181800,WMO_GRIB2,latLon,1440:721:90N:0E:90S:359.75E:0.25:0.25,None,s^-1,wind_speed_shear,05cf5b56-0f86-4819-b713-1272b97b06c5,30-hour Forecast,"[{'level_value': '0', 'level_description': 'Tr..."
11418,U FLX,"Momentum flux, u-component",201501241800,202606270900,WMO_GRIB2,latLon,1440:721:90N:0E:90S:359.75E:0.25:0.25,None,N m^-2,downward_eastward_momentum_flux_in_air,NaN,3-hour Average (initial+234 to initial+237),"[{'level_value': '0', 'level_description': 'Gr..."


#### Now, you can filter to find more information about the contents of the dataset.

For example, let's say I want to just get the parameters that are available at 500mb.

In [48]:
params_500mb = [
    p for p in all_metadata
    if any(
        lvl.get("level_description") == "Isobaric surface" and
        lvl.get("level_value") == "500"
        for lvl in p.get("levels", [])
    )]

#params_500mb
set([x['param_description'] for x in params_500mb])

{'5-wave geopotential height',
 'Absolute vorticity',
 'Cloud water mixing ratio',
 'Geopotential height',
 'Graupel',
 'Ice water mixing ratio',
 'Icing severity',
 'Ozone mixing ratio',
 'Rain water mixing ratio',
 'Relative humidity',
 'Snow water mixing ratio',
 'Specific humidity',
 'Temperature',
 'Total cloud cover',
 'Vertical velocity (geometric)',
 'Vertical velocity (pressure)',
 'u-component of wind',
 'v-component of wind'}

## Dataset Volume and Related Datasets

`GET /datasets/{dsid}/volume/` reports total archive size. `GET /datasets/{dsid}/related_datasets/` lists companion datasets.


In [ ]:
volume  = requests.get(f"{BASE_URL}/datasets/{DSID}/volume/").json()
related = requests.get(f"{BASE_URL}/datasets/{DSID}/related_datasets/").json()
print("Total volume:", volume["data"]["total_volume"])


---

## Summary

GDEX provides dedicated endpoints for every facet of dataset metadata — from abstract text and spatial bounds to variable lists and archive size — all following the same response envelope.


:::{seealso}
{doc}`api_subset` — use the metadata discovered here to build and submit a targeted subset request.
:::


## Resources and References

- [GDEX API Documentation](https://api.gdex.ucar.edu/documentation/)
- [GDEX Dataset Catalog](https://gdex.ucar.edu/datasets/)
